# 04 – Hypothesis testing
Formal statistical tests for the research questions.

In [1]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from scipy import stats
from scipy.stats.contingency import association
from pathlib import Path

clean_folder = Path("../data/clean")
daily = pd.read_csv(clean_folder / "daily_summer.csv", parse_dates=["date"])

daily = daily.sort_values("date").reset_index(drop=True)
daily["prev_day_hot"] = daily.groupby("year")["hot_day"].shift(1)

summer = daily[(daily["in_summer"] == 1) & daily["ems_missions"].notna()].copy()
summer["ems_index"] = summer["ems_missions"] / summer.groupby("year")["ems_missions"].transform("mean") * 100
summer["cpr_index"] = summer["cpr_alert_missions"] / summer.groupby("year")["cpr_alert_missions"].transform("mean") * 100
summer["temp_band"] = pd.cut(summer["tmax"], bins=[0, 20, 22, 24, 26, 28, 30, 32, 34, 45], right=False,
                             labels=["<20", "20–22", "22–24", "24–26", "26–28", "28–30", "30–32", "32–34", "34+"])

alpha = 0.05
print(summer.shape)

(822, 17)


In [2]:
def compare_groups(group_a, group_b, name_a, name_b, alternative="greater"):
    t_stat, p_value = stats.ttest_ind(group_a, group_b, equal_var=False, alternative=alternative)

    diff = group_a.mean() - group_b.mean()
    se = np.sqrt(group_a.var(ddof=1) / len(group_a) + group_b.var(ddof=1) / len(group_b))
    ci_low, ci_high = diff - 1.96 * se, diff + 1.96 * se

    print(f"{name_a}: n = {len(group_a)}, mean = {group_a.mean():.1f}")
    print(f"{name_b}: n = {len(group_b)}, mean = {group_b.mean():.1f}")
    print(f"Difference: {diff:.1f} index points, 95% CI [{ci_low:.1f}, {ci_high:.1f}]")
    print(f"t = {t_stat:.2f}, p-value = {p_value:.2e}")

    if p_value < alpha:
        print("Decision: reject H0")
    else:
        print("Decision: we cannot reject H0")

In [3]:
def report(model, variable, label):
    ci_low, ci_high = model.conf_int().loc[variable]
    print(f"{label}: {model.params[variable]:+.2f} index points, "
          f"95% CI [{ci_low:.2f}, {ci_high:.2f}], p = {model.pvalues[variable]:.2e}")

hac = {"cov_type": "HAC", "cov_kwds": {"maxlags": 7}}
summer = summer.sort_values("date")

### Test 1 – Hot days vs. normal days
H0: mean EMS demand on hot days ≤ normal days
H1: mean EMS demand on hot days > normal days (one-sided, α = 0.05)

In [4]:
hot = summer[summer["hot_day"] == 1]["ems_index"]
normal = summer[summer["hot_day"] == 0]["ems_index"]

print("Levene p-value:", round(stats.levene(hot, normal).pvalue, 3))
print("Shapiro p-value, hot days:   ", round(stats.shapiro(hot).pvalue, 3))
print("Shapiro p-value, normal days:", round(stats.shapiro(normal).pvalue, 3))

Levene p-value: 0.731
Shapiro p-value, hot days:    0.038
Shapiro p-value, normal days: 0.077


In [5]:
compare_groups(hot, normal, "Hot days", "Normal days")

pooled_sd = np.sqrt(((len(hot) - 1) * hot.var(ddof=1) + (len(normal) - 1) * normal.var(ddof=1)) / (len(hot) + len(normal) - 2))
print(f"Cohen's d = {(hot.mean() - normal.mean()) / pooled_sd:.2f}")

print("Mann-Whitney U p-value:", f"{stats.mannwhitneyu(hot, normal, alternative='greater').pvalue:.2e}")

Hot days: n = 155, mean = 105.5
Normal days: n = 667, mean = 98.7
Difference: 6.7 index points, 95% CI [5.7, 7.8]
t = 12.20, p-value = 5.26e-27
Decision: reject H0
Cohen's d = 1.09
Mann-Whitney U p-value: 1.46e-28


### Test 2 – Consistency across summers (sign test)
Consecutive hot days are not independent (heatwaves), so the t-test p-value is likely too optimistic.
A simpler check treats each summer as one observation.
H0: hot days are busier in only half of the summers (pure chance)
H1: hot days are busier in more summers than chance would explain

In [6]:
per_year = summer.groupby(["year", "hot_day"])["ems_index"].mean().unstack()
per_year.columns = ["normal_days", "hot_days"]
per_year["hot_busier"] = per_year["hot_days"] > per_year["normal_days"]
print(per_year.round(1))

k = per_year["hot_busier"].sum()
n = len(per_year)
p_sign = stats.binomtest(k, n, p=0.5, alternative="greater").pvalue
print(f"\nHot days busier in {k} of {n} summers, p = {p_sign:.4f}")

      normal_days  hot_days  hot_busier
year                                   
2018         97.9     105.7        True
2019         97.8     105.9        True
2020         98.6     106.6        True
2021         99.1     105.3        True
2022         99.0     103.3        True
2023         99.1     106.8        True
2024         99.1     105.0        True
2025         99.4     104.2        True
2026         98.2     106.4        True

Hot days busier in 9 of 9 summers, p = 0.0020


### Model 1 – Adjusted hot-day effect
The t-tests treat every day as independent, but hot days come in streaks, and demand also varies by year, month and weekday.
This regression compares days **within the same year, month and weekday**, and uses HAC standard errors, which correct for streaks of similar days.

In [7]:
main_model = smf.ols(
    "ems_index ~ hot_day + C(year) + C(month) + C(weekday_num)",
    data=summer
).fit(**hac)

report(main_model, "hot_day", "Hot day (adjusted)")
print(f"R-squared: {main_model.rsquared:.2f}")

temp_model = smf.ols(
    "ems_index ~ tmax + C(year) + C(month) + C(weekday_num)",
    data=summer
).fit(**hac)

report(temp_model, "tmax", "Per +1°C")

Hot day (adjusted): +7.06 index points, 95% CI [5.89, 8.23], p = 2.95e-32
R-squared: 0.32
Per +1°C: +0.84 index points, 95% CI [0.71, 0.97], p = 3.19e-36


### Test 3 – Correlation between temperature and demand
H0: no correlation between daily maximum temperature and EMS demand (r = 0)
H1: there is a correlation

In [8]:
r, p_r = stats.pearsonr(summer["tmax"], summer["ems_index"])
rho, p_rho = stats.spearmanr(summer["tmax"], summer["ems_index"])

print(f"Pearson r = {r:.2f}, p = {p_r:.2e}")
print(f"Spearman rho = {rho:.2f}, p = {p_rho:.2e}")

Pearson r = 0.51, p = 1.52e-56
Spearman rho = 0.49, p = 2.11e-51


### Test 4 – Differences across temperature bands
H0: mean EMS demand is the same in all temperature bands
H1: at least one band differs

In [9]:
band_groups = [group["ems_index"] for band, group in summer.groupby("temp_band", observed=True)]

f_stat, p_anova = stats.f_oneway(*band_groups)
h_stat, p_kruskal = stats.kruskal(*band_groups)

print(f"ANOVA: F = {f_stat:.1f}, p = {p_anova:.2e}")
print(f"Kruskal-Wallis: H = {h_stat:.1f}, p = {p_kruskal:.2e}")

ANOVA: F = 33.9, p = 2.67e-46
Kruskal-Wallis: H = 198.7, p = 1.20e-38


### Test 5 – Hot days and weekdays
Sundays are about 4% quieter (EDA). If hot days fell mostly on Sundays or mostly on weekdays, Test 1 would be biased.
H0: hot days are independent of the weekday
H1: hot days are more common on some weekdays

In [10]:
weekday_table = pd.crosstab(summer["weekday"], summer["hot_day"])
print(weekday_table)

chi2, p_chi, dof, expected = stats.chi2_contingency(weekday_table)
print(f"\nChi-square = {chi2:.2f}, p = {p_chi:.2f}, Cramér's V = {association(weekday_table, method='cramer'):.2f}")

hot_day      0   1
weekday           
Friday      94  23
Monday     100  17
Saturday    94  25
Sunday      94  23
Thursday    92  24
Tuesday    100  18
Wednesday   93  25

Chi-square = 3.56, p = 0.74, Cramér's V = 0.07


### Test 6 – Hot days without a heat warning vs. normal days
H0: mean demand on hot days without a warning ≤ normal days without a warning
H1: hot days without a warning are busier

In [11]:
hot_no_warning = summer[(summer["hot_day"] == 1) & (summer["heat_warning"] == 0)]["ems_index"]
normal_no_warning = summer[(summer["hot_day"] == 0) & (summer["heat_warning"] == 0)]["ems_index"]

compare_groups(hot_no_warning, normal_no_warning, "Hot, no warning", "Normal days")

Hot, no warning: n = 72, mean = 103.8
Normal days: n = 660, mean = 98.7
Difference: 5.1 index points, 95% CI [3.7, 6.6]
t = 6.97, p-value = 2.66e-10
Decision: reject H0


### Test 7 – Warned vs. unwarned hot days (30–34°C)
This exploratory test compares only days between 30 and 34°C. Narrowing the range reduces temperature differences but doesn't remove them: warned days are still about 1°C hotter on average.
Any difference may therefore reflect remaining temperature differences, warm nights, consecutive hot days or other factors. A significant unadjusted difference does not show an independent warning effect – see Model 2.
H0: no difference between warned and unwarned days
H1: there is a difference (two-sided)

In [12]:
band_30_34 = summer[(summer["tmax"] >= 30) & (summer["tmax"] < 34)]

print(band_30_34.groupby("heat_warning").agg(
    days=("tmax", "size"),
    avg_tmax=("tmax", "mean"),
    avg_index=("ems_index", "mean")
).round(1))

compare_groups(band_30_34[band_30_34["heat_warning"] == 1]["ems_index"],
               band_30_34[band_30_34["heat_warning"] == 0]["ems_index"],
               "Warning", "No warning", alternative="two-sided")

              days  avg_tmax  avg_index
heat_warning                           
0               70      31.1      103.8
1               49      32.2      106.3
No warning: n = 70, mean = 103.8
Difference: 2.5 index points, 95% CI [0.4, 4.7]
t = 2.28, p-value = 2.49e-02
Decision: reject H0


### Model 2 – Does a heat warning add information beyond temperature?
Adjusts for daily maximum temperature (including a curved effect via `tmax²`), year, month and weekday.
This measures an association, not the causal effect of issuing a warning.

In [13]:
warning_model = smf.ols(
    "ems_index ~ tmax + I(tmax**2) + heat_warning + C(year) + C(month) + C(weekday_num)",
    data=summer
).fit(**hac)

report(warning_model, "heat_warning", "Heat warning (adjusted for temperature)")

Heat warning (adjusted for temperature): +1.27 index points, 95% CI [-0.82, 3.36], p = 2.32e-01


### Test 8 – The day after a hot day
Only days that are not hot themselves, so heatwave streaks are not counted twice.
H0: demand on the first non-hot day after a hot day ≤ other normal days
H1: it is higher

In [14]:
after_hot = summer[(summer["hot_day"] == 0) & (summer["prev_day_hot"] == 1)]["ems_index"]
other_normal = summer[(summer["hot_day"] == 0) & (summer["prev_day_hot"] == 0)]["ems_index"]

compare_groups(after_hot, other_normal, "Day after a hot day", "Other normal days")

Day after a hot day: n = 75, mean = 101.1
Other normal days: n = 592, mean = 98.4
Difference: 2.6 index points, 95% CI [0.9, 4.4]
t = 2.90, p-value = 2.35e-03
Decision: reject H0


### Model 3 – The day after a hot day, adjusted
Only non-hot days. Adjusts for today's temperature, year, month and weekday.

In [15]:
lag_subset = summer[(summer["hot_day"] == 0) & summer["prev_day_hot"].notna()]

lag_model = smf.ols(
    "ems_index ~ prev_day_hot + tmax + C(year) + C(month) + C(weekday_num)",
    data=lag_subset
).fit(**hac)

report(lag_model, "prev_day_hot", "Day after a hot day (adjusted)")

Day after a hot day (adjusted): +1.63 index points, 95% CI [0.10, 3.16], p = 3.64e-02


### Test 9 – CPR-alert dispatches on hot days
Compares the annual-normalised number of EMS missions dispatched with a CPR alert (suspected need for resuscitation at the time of the call).
These are dispatch alerts, **not confirmed cardiac arrests**.
H0: CPR-alert demand on hot days ≤ normal days
H1: it is higher

In [16]:
compare_groups(summer[summer["hot_day"] == 1]["cpr_index"],
               summer[summer["hot_day"] == 0]["cpr_index"],
               "CPR alerts, hot days", "CPR alerts, normal days")

CPR alerts, hot days: n = 155, mean = 105.2
CPR alerts, normal days: n = 667, mean = 98.8
Difference: 6.5 index points, 95% CI [1.5, 11.4]
t = 2.55, p-value = 5.73e-03
Decision: reject H0


### Model 4 – Sensitivity: does the result depend on 2026?
2026 includes the record heatwave and the seven highest demand values. The main model is re-run without 2026, and without only the extreme week.

In [17]:
scenarios = {
    "All years": summer,
    "Excluding 2026": summer[summer["year"] < 2026],
    "Excluding 26 Jun – 2 Jul 2026": summer[~summer["date"].between("2026-06-26", "2026-07-02")]
}

for name, subset in scenarios.items():
    model = smf.ols(
        "ems_index ~ hot_day + C(year) + C(month) + C(weekday_num)",
        data=subset
    ).fit(**hac)
    report(model, "hot_day", name)

All years: +7.06 index points, 95% CI [5.89, 8.23], p = 2.95e-32
Excluding 2026: +6.91 index points, 95% CI [5.79, 8.03], p = 1.36e-33
Excluding 26 Jun – 2 Jul 2026: +6.74 index points, 95% CI [5.68, 7.79], p = 5.76e-36


## Summary of tests and models

| # | Question | Method | Result | Interpretation |
|---|---|---|---|---|
| 1 | Hot days busier? | Welch t-test (one-sided) | +6.7 pts, 95% CI [5.7, 7.8], d = 1.09 | Reject H0 (unadjusted) |
| 2 | Consistent across years? | Sign test | 9 of 9 summers, p = 0.002 | Consistent; does not rule out confounding |
| M1 | Hot days, adjusted | Regression (year, month, weekday; HAC) | **+7.1 pts, 95% CI [5.9, 8.2]** | Robust main result |
| M1 | Per +1°C, adjusted | Regression | **+0.84 pts, 95% CI [0.71, 0.97]** | Gradual rise with temperature |
| 3 | Correlation with temperature | Pearson / Spearman | r = 0.51 / 0.49 | Association, not causation |
| 4 | Differences across temperature bands | ANOVA / Kruskal-Wallis | p < 0.001 | No precise threshold identified |
| 5 | Hot days spread across weekdays | Chi-square | p = 0.74 | No significant difference detected; residual confounding possible |
| 6 | Hot days without warning busier? | Welch t-test | +5.1 pts, 95% CI [3.7, 6.6] | Reject H0 (unadjusted) |
| 7 | Warned vs. unwarned, 30–34°C | Welch t-test (two-sided) | +2.5 pts, 95% CI [0.4, 4.7], p = 0.025 | Temperature-confounded; see M2 |
| M2 | Warning, adjusted for temperature | Regression | +1.3 pts, 95% CI [−0.8, 3.4], p = 0.23 | **Not significant** |
| 8 | Day after a hot day | Welch t-test | +2.6 pts, 95% CI [0.9, 4.4] | Unadjusted; see M3 |
| M3 | Day after a hot day, adjusted | Regression | +1.6 pts, 95% CI [0.1, 3.2], p = 0.04 | Weak, indicative only |
| 9 | CPR-alert dispatches | Welch t-test | +6.5 pts, 95% CI [1.5, 11.4], p = 0.006 | Exploratory; dispatch alerts, not confirmed cases |
| M4 | Excluding 2026 / extreme week | Regression | +6.9 / +6.7 pts | Result does not depend on 2026 |

**Caveats**
- The t-tests assume independent days; hot days come in streaks, so their p-values and confidence intervals are too optimistic. The regressions with HAC errors and the yearly sign test are the more reliable evidence.
- Multiple testing: a conservative Bonferroni threshold for nine tests is 0.05 / 9 ≈ 0.0056. Test 7 (p ≈ 0.025), Test 9 (p ≈ 0.0057) and Model 3 (p ≈ 0.04) do not meet it and should be read as indicative.
- All results are associations from observational data, not proof of causation.

## Main findings

**Higher daily maximum temperatures are associated with higher emergency medical demand in Berlin.**

1. **Hot days show higher EMS demand.** On days of at least 30°C, demand is about 7% above an average summer day: +6.7 index points unadjusted, and **+7.1 points (95% CI 5.9–8.2)** after adjusting for year, month and weekday, with standard errors that account for heatwave streaks.
2. **The pattern is consistent and robust.** Hot days had higher demand in all nine summers from 2018 to 2026 (sign test p = 0.002). The adjusted estimate barely changes when 2026 (+6.9) or the extreme June/July 2026 heatwave (+6.7) is excluded.
3. **Demand rises gradually with temperature.** Each additional °C of daily maximum temperature is associated with about **+0.8 index points** (95% CI 0.7–1.0). Demand ranges from about 95 index points below 20°C to about 108 above 34°C; there is no sharp threshold at 30°C.
4. **Heat warnings show no statistically significant additional association with EMS demand after adjusting for temperature and calendar effects. The estimated difference is +1.3 index points (95% CI −0.8 to 3.4; p = 0.23).** 72 of 155 hot days had no DWD heat warning, yet still showed about 5% higher demand (unadjusted). DWD warnings follow different, health-oriented criteria (perceived temperature over several days), so this is not a failure of the warning system. For demand planning, daily temperature is the stronger signal.
5. **A next-day effect is weak and uncertain.** On non-hot days following a hot day, demand is slightly higher (+1.6 points adjusted, 95% CI 0.1–3.2, p = 0.04), which does not pass a conservative multiple-testing threshold.
6. **CPR-alert dispatches are a secondary, exploratory outcome.** They are higher on hot days (+6.5 points, 95% CI 1.5–11.4), but these are dispatch alerts, not confirmed cardiac arrests, and the result does not pass a conservative multiple-testing threshold.

**Interpretation for emergency-service planning:** daily maximum temperature is a simple, robust indicator of higher emergency medical demand, about +0.8% per °C. This becomes more relevant as climate change brings more hot days to Berlin.

**Limitations**
- Observational data: associations, not proof of causation.
- One weather station (Tempelhof) represents the whole city.
- Daily maximum temperature does not capture all heat exposure (e.g. warm nights).
- Five summer days are missing in 2026; one suspected reporting anomaly (7 June 2018) was excluded; 2026 weather data is preliminary.
- Nine exploratory tests increase the risk of false positives; weaker results (Tests 7–9, Model 3) are indicative only.